# Risk Framework Tutorial

This tutorial demonstrates QuantStrata's risk infrastructure: Value-at-Risk (VaR), Greeks aggregation, and stress testing.

**Topics covered:**
- Historical, Parametric, and Monte Carlo VaR
- Greeks aggregation and risk-factor decomposition
- Multi-factor stress scenarios (CompositeShock) and preset/historical scenario generation
- Using the VaR facade (compute_var)

In [1]:
# Standard imports
import sys
sys.path.insert(0, '../../..')

import numpy as np
import matplotlib.pyplot as plt

plt.style.use('seaborn-v0_8-whitegrid')
np.random.seed(42)

print("Setup complete.")

Setup complete.


## 1. Historical VaR

Historical VaR uses a P&L (or return) time series. VaR is the negative of the (1 - confidence) quantile of P&L.

In [2]:
from src.risk.var import historical_var, VarConfig

# Simulate 252 days of daily P&L (e.g. from revaluation or backtest)
pnl_series = np.random.randn(252) * 10_000

config = VarConfig(confidence=0.99, horizon_days=1, method="historical")
result = historical_var(pnl_series, config)

print(f"99% 1-day VaR: {result.var:,.2f}")
print(f"CVaR (expected shortfall): {result.cvar:,.2f}")
print(f"Metadata: {result.metadata}")

99% 1-day VaR: 19,733.41
CVaR (expected shortfall): 22,108.19
Metadata: {'n_observations': 252}


## 2. Parametric VaR (delta-normal)

Parametric VaR uses portfolio sensitivities and factor volatilities. VaR = z(confidence) * sqrt(Gamma' Sigma Gamma); scaled by sqrt(horizon_days).

In [3]:
from src.risk.var import parametric_var, VarConfig
from src.risk.sensitivities.result import SensitivityKey, SensitivityRow, SensitivitiesReport
from src.marketdata.core.ids import MarketId

# Minimal example: synthetic sensitivities and factor vols (1% daily per factor)
mid = MarketId("FX", "SPOT", "EURUSD")
key = SensitivityKey(greek="delta", market_id=mid)
report = SensitivitiesReport(rows=[
    SensitivityRow(key=key, value=100_000.0, method="analytic", bump=None, units="per 1 spot"),
])
factor_volatilities = {key: 0.01}
config = VarConfig(confidence=0.99, horizon_days=1, method="parametric")
result = parametric_var(report, factor_volatilities, config)

print(f"99% 1-day Parametric VaR: {result.var:,.2f}")
print(f"CVaR: {result.cvar:,.2f}")

99% 1-day Parametric VaR: 2,326.35
CVaR: 2,665.21


In [4]:
# With a real portfolio: report = compute_sensitivities(portfolio, market, portfolio_pricer, ...)
# then factor_volatilities = {row.key: daily_vol for row in report.rows}

## 3. Greeks aggregation

Aggregate sensitivities by greek and by risk factor (spot, vol, rate).

In [5]:
from src.risk.sensitivities.aggregation import aggregate_sensitivities, GreeksSummary
from src.risk.sensitivities.result import SensitivitiesReport, SensitivityRow, SensitivityKey
from src.marketdata.core.ids import MarketId

# Build a small report (e.g. from compute_sensitivities)
mid = MarketId("FX", "SPOT", "EURUSD")
rows = [
    SensitivityRow(SensitivityKey("delta", mid), 50_000.0, "analytic", None, "per 1 spot"),
    SensitivityRow(SensitivityKey("vega", MarketId("FX", "VOL", "EURUSD")), 20_000.0, "analytic", None, "per 1 vol"),
]
report = SensitivitiesReport(rows=rows)
summary = aggregate_sensitivities(report, include_per_market_id=True)

print("Totals by greek:", summary.totals_by_greek)
print("Totals by risk factor:", summary.totals_by_factor)
print("Per market_id (first 3):", summary.per_market_id[:3])

Totals by greek: {'delta': 50000.0, 'vega': 20000.0}
Totals by risk factor: {'spot': 50000.0, 'vol': 20000.0}
Per market_id (first 3): [(MarketId(asset_class='FX', mkt_type='SPOT', name='EURUSD', qualifiers=()), 'delta', 50000.0), (MarketId(asset_class='FX', mkt_type='VOL', name='EURUSD', qualifiers=()), 'vega', 20000.0)]


## 4. Stress testing: CompositeShock and preset packs

Multi-factor scenarios apply several shocks in sequence. Use CompositeShock or preset_stress_pack.

In [6]:
from src.marketdata.scenarios.shocks import CompositeShock, SpotShock, VolShock, ParallelRateShock
from src.risk.scenarios.generation import preset_stress_pack
from src.marketdata.core.ids import MarketId

spot_id = MarketId("FX", "SPOT", "EURUSD")
vol_id = MarketId("FX", "VOL", "EURUSD")
domestic_curve_id = MarketId("IR", "CURVE", "USD_OIS")

# Preset pack (e.g. crisis_style = spot -15%, vol +30%, rates -50bp)
pack = preset_stress_pack(
    "crisis_style",
    spot_id=spot_id,
    vol_id=vol_id,
    domestic_curve_id=domestic_curve_id,
)
print("Preset scenarios:", list(pack.scenarios.keys()))
print("crisis_style is a CompositeShock:", type(list(pack.scenarios.values())[0]).__name__)

Preset scenarios: ['crisis_style']
crisis_style is a CompositeShock: CompositeShock


## 5. VaR facade: compute_var

Use compute_var to dispatch by config.method (historical, parametric, mc).

In [7]:
from src.risk.var import compute_var, VarConfig

# Historical
config = VarConfig(method="historical", confidence=0.95)
result = compute_var(config, pnl_series=np.random.randn(100) * 5_000)
print(f"95% Historical VaR: {result.var:,.2f}")

95% Historical VaR: 7,152.14


## Summary

- **Historical VaR:** P&L series -> quantile; no distribution assumption.
- **Parametric VaR:** Sensitivities + factor vols -> delta-normal VaR.
- **Monte Carlo VaR:** Full revaluation under simulated factor shocks (see reference doc for DiagonalFactorModel).
- **Greeks aggregation:** aggregate_sensitivities(report) -> GreeksSummary (totals by greek and by risk factor).
- **Stress testing:** CompositeShock for multi-factor; preset_stress_pack for predefined scenarios; shocks_from_historical_series for historical-based shocks.

See [Risk Infrastructure Reference](../../reference/risk/risk_infrastructure.md) and [Risk Framework Guide](../../guides/risk/risk_framework.md) for full API details.